In [12]:
import pandas as pd
import load_TU_data
from otp_client import get_all_routes_for_mode, load_all_candidates
from find_similar_trip import find_similar_trip
from otp_utils import has_invalid_route_name, resolve_route_short_names, get_via_stops
from tu_gtfs_stations_match import match_tu_gtfs_stations
import importlib

In [13]:
print("Loading TU data...")
data_dir = "/home/simpal/O/TU_Rejseplan/Data/TU/"
tu_session, tu_tur, tu_deltur, tu_stations = load_TU_data.load_tu(
    data_dir=data_dir,
    session_file="tu_session_secret_2015_2025.xlsx",
    tur_file="tu_tur_secret_2015_2025.xlsx",
    deltur_file="tu_deltur_2015_2025.xlsx",
    stations_file="Stationer_tudatabase.xlsx"
)
print("TU data loaded")

#Configurartion
mode_map = {
    #TU: OTP
    31: "BUS",
    32: "S_TRAIN",
    33: "RAIL",
    34: "SUBWAY",
    37: "TRAM",
    41: "FERRY",
    35: "BUS"
}
otp_url = "http://localhost:8080/otp/gtfs/v1"
search_window = "PT30M"
print(f"Search window: {search_window}")

# RAIL, TRAM and SUBWAY are missing route name in TU.
# So taking all routes for these modes. Which will be used when modes
# that do include route name in TU only can access those routes, but for
# those that do not, all routes will be used.
otp_mode_routes_cache = {mode: get_all_routes_for_mode(mode) for mode in ["RAIL", "TRAM", "SUBWAY", "FERRY"]}

#Small processing of TU data
tu_tur = tu_tur[tu_tur["PtPrimMode"].isin([31, 32, 33, 34, 37, 41])]
tu_tur = tu_tur[(tu_tur["DiaryYear"] == 2024) & (tu_tur["DiaryMonth"] == 6)]
tu_deltur = tu_deltur[tu_deltur["TurId"].isin(tu_tur["TurId"])]
tu_deltur["otp_mode"] = tu_deltur["StageMode"].map(mode_map)

#Map TU and GTFS stations
tu_gtfs_station_df = match_tu_gtfs_stations(
    tu_stations,
    period=(tu_tur["DiaryDate"].min(), tu_tur["DiaryMonth"].max()),
    bbox_buffer_m=1000,
    name_match_threshold=0.6
)
print("Mapping of TU and GTFS station has been exported to", data_dir + "tu_gtfs_station_df.csv")
tu_gtfs_station_df.to_csv(data_dir + "tu_gtfs_station_df.csv")


Loading TU data...
TU data loaded
Search window: PT30M
Mapping of TU and GTFS station has been exported to /home/simpal/O/TU_Rejseplan/Data/TU/tu_gtfs_station_df.csv


In [51]:
#This is where for loop in main starts
i_TurId = 2708293
tu_tur_row = tu_tur[tu_tur["TurId"] == i_TurId].iloc[0]

tu_deltur_sub = tu_deltur.loc[tu_deltur["TurId"] == i_TurId]
tu_tur_row[["orig_lat","orig_lon","tiladrlat","tiladrlon"]]

orig_lat     55.684119
orig_lon     12.587749
tiladrlat    55.634051
tiladrlon    12.066489
Name: 274001, dtype: object

In [52]:
tu_tur_row["depart_dt_str"]

'2024-06-04T21:15:00+0200'

In [53]:
print("\n\n____________________________________________________________________________________________")
print(f"TurId: {i_TurId}. With SessionId: {tu_tur_row['SessionId']}.")
print(f"Tur coordinates origin (lat lon) :     {tu_tur_row['tiladrlat']} {tu_tur_row['tiladrlon']}.")
print(f"Tur coordinates destination (lat lon): {tu_tur_row['tiladrlat']} {tu_tur_row['tiladrlon']}.")
# Print for debugging
tu_deltur_sub_print_col = ["StageMode", "StageLength", "StageWaitMin", "StageDurationMin", "Route","FromStation", "ToStation"]
print("tu_deltur_sub:")
print(tu_deltur_sub[tu_deltur_sub_print_col].to_string(index=False, max_colwidth=None))

route_names, route_names_ext, modes_json, modes_list = resolve_route_short_names(
    tu_deltur_sub,
    mode_map,
    otp_mode_routes_cache
)
print(f"route_short_name: {route_names}")
print(f"route_names_ext: {route_names_ext}")

if not modes_json:
    print(f"No valid public transport modes found for TurId: {i_TurId}")

print(f"modes_json: {modes_json}")
if any(mode in ["BUS", "S_TRAIN"] for mode in modes_list) and not route_names:
    print(f"No valid route found for TurId: {i_TurId}")

if has_invalid_route_name(route_names) and route_names:
    print(f"Invalid route name: {route_names}")



____________________________________________________________________________________________
TurId: 2708293. With SessionId: 523188.
Tur coordinates origin (lat lon) :     55.63405106095683 12.066488731223806.
Tur coordinates destination (lat lon): 55.63405106095683 12.066488731223806.
tu_deltur_sub:
 StageMode  StageLength  StageWaitMin  StageDurationMin Route    FromStation   ToStation
         1          0.6           NaN              10.0   NaN            NaN         NaN
        34          1.9           5.0               3.0   NaN Kongens Nytorv København H
        33         31.3          13.0              22.0   NaN    København H    Roskilde
         2          2.0           NaN              15.0   NaN            NaN         NaN
route_short_name: []
route_names_ext: []
modes_json: [{'mode': 'SUBWAY'}, {'mode': 'RAIL'}]


In [54]:
#Get the gtfs stop_ids for stations respondent travel through
via_stopids = get_via_stops(tu_deltur_sub=tu_deltur_sub, tu_gtfs_station_df=tu_gtfs_station_df)
via_stopids

['1:20240102_000008603308',
 '1:20240102_000008603330',
 '1:20240102_000008600626',
 '1:20240102_000008600617']

In [55]:
# 2. Fetch all candidates (handles pagination & concat internally)
is_bus_s_train = any(mode in ["BUS", "S_TRAIN"] for mode in modes_list)
print(is_bus_s_train)
is_rail_tram_subway_ferry = any(mode in ["RAIL", "SUBWAY", "TRAM", "FERRY"] for mode in modes_list)
print(is_rail_tram_subway_ferry)

False
True


In [57]:
import importlib
import otp_parser
import otp_client

importlib.reload(otp_parser)
importlib.reload(otp_client)

from otp_client import load_all_candidates
if is_bus_s_train and is_rail_tram_subway_ferry:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=route_names_ext,
        via_stopids=via_stopids,
        search_window=search_window,
        otp_url=otp_url)

elif is_bus_s_train:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=route_names,
        via_stopids=via_stopids,
        search_window=search_window,
        otp_url=otp_url,)
elif is_rail_tram_subway_ferry:
    otp_candidates_df = load_all_candidates(
        tu_tur_row=tu_tur_row,
        modes_json=modes_json,
        route_short_name=None,
        via_stopids=via_stopids,
        search_window=search_window,
        otp_url=otp_url)
otp_candidates_df

,iteration_id,leg_id,start_trip,end_trip,start_leg,end_leg,mode,route_short_name,distance_km,duration_min,waiting_time_min,from,to,generalized_cost,leg_geometry,system_notice_tag,system_notice_text,start_dt
0,-6,0,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524221000,1717524900000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:03:41+02:00
1,-6,1,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524900000,1717525140000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:03:41+02:00
2,-6,2,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525140000,1717525455000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:03:41+02:00
3,-6,3,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525740000,1717527000000,RAIL,IC,31.208,21,0.0,København H,Roskilde St.,2145,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:03:41+02:00
4,-6,4,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717527000000,1717528412000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:03:41+02:00
5,-5,0,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717525421000,1717526100000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:23:41+02:00
6,-5,1,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526100000,1717526340000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:23:41+02:00
7,-5,2,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526340000,1717526655000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:23:41+02:00
8,-5,3,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526820000,1717528020000,RAIL,RE,31.426,20,0.0,København H,Roskilde St.,1965,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:23:41+02:00
9,-5,4,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717528020000,1717529432000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:23:41+02:00


In [58]:
otp_candidates_df = otp_candidates_df.sort_values(
    ["iteration_id", "start_leg"]
).reset_index(drop=True)

otp_candidates_df["waitingtime"] = (
    otp_candidates_df["start_leg"]
    - otp_candidates_df.groupby("iteration_id")["end_leg"].shift()
) / 60 / 1000


In [59]:
otp_candidates_df

,iteration_id,leg_id,start_trip,end_trip,start_leg,end_leg,mode,route_short_name,distance_km,duration_min,waiting_time_min,from,to,generalized_cost,leg_geometry,system_notice_tag,system_notice_text,start_dt,waitingtime
0,-6,0,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524221000,1717524900000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:03:41+02:00,NaN
1,-6,1,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524900000,1717525140000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:03:41+02:00,0.00
2,-6,2,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525140000,1717525455000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:03:41+02:00,0.00
3,-6,3,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525740000,1717527000000,RAIL,IC,31.208,21,0.0,København H,Roskilde St.,2145,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:03:41+02:00,4.75
4,-6,4,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717527000000,1717528412000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:03:41+02:00,0.00
5,-5,0,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717525421000,1717526100000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:23:41+02:00,NaN
6,-5,1,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526100000,1717526340000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:23:41+02:00,0.00
7,-5,2,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526340000,1717526655000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:23:41+02:00,0.00
8,-5,3,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526820000,1717528020000,RAIL,RE,31.426,20,0.0,København H,Roskilde St.,1965,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:23:41+02:00,2.75
9,-5,4,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717528020000,1717529432000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:23:41+02:00,0.00


In [ ]:
if otp_candidates_df.empty:
    print(f"No OTP trips found for TurId: {i_TurId}")

In [ ]:
route_names

In [60]:
#TODO: Move from main():
if "route_short_name" not in otp_candidates_df.columns:
    print(f"OTP candidates are missing route_short_name for TurId: {i_TurId}")

In [61]:
required_routes = set(map(str, route_names))

iteration_ids_with_required_routes = (
    otp_candidates_df.groupby("iteration_id")["route_short_name"]
    .apply(
        lambda routes: required_routes.issubset(
            set(routes.dropna().astype(str))
        )
    )
)
iteration_ids_with_required_routes

iteration_id
-6    True
-5    True
-4    True
-3    True
-2    True
-1    True
 0    True
 1    True
 2    True
 3    True
Name: route_short_name, dtype: bool

In [62]:
otp_candidates_df = otp_candidates_df[
    otp_candidates_df["iteration_id"].isin(
        iteration_ids_with_required_routes[
            iteration_ids_with_required_routes
        ].index
    )
].reset_index(drop=True)

if otp_candidates_df.empty:
    print(f"No OTP trips include all required BUS/S_TRAIN routes for TurId: {i_TurId}")
otp_candidates_df

,iteration_id,leg_id,start_trip,end_trip,start_leg,end_leg,mode,route_short_name,distance_km,duration_min,waiting_time_min,from,to,generalized_cost,leg_geometry,system_notice_tag,system_notice_text,start_dt,waitingtime
0,-6,0,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524221000,1717524900000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:03:41+02:00,NaN
1,-6,1,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524900000,1717525140000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:03:41+02:00,0.00
2,-6,2,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525140000,1717525455000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:03:41+02:00,0.00
3,-6,3,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525740000,1717527000000,RAIL,IC,31.208,21,0.0,København H,Roskilde St.,2145,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:03:41+02:00,4.75
4,-6,4,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717527000000,1717528412000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:03:41+02:00,0.00
5,-5,0,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717525421000,1717526100000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:23:41+02:00,NaN
6,-5,1,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526100000,1717526340000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:23:41+02:00,0.00
7,-5,2,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526340000,1717526655000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:23:41+02:00,0.00
8,-5,3,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526820000,1717528020000,RAIL,RE,31.426,20,0.0,København H,Roskilde St.,1965,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:23:41+02:00,2.75
9,-5,4,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717528020000,1717529432000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:23:41+02:00,0.00


In [63]:
# Ensure all transit modes from the TU data are used in the OTP itinerary

if "mode" not in otp_candidates_df.columns:
    print(f"OTP candidates are missing mode for TurId: {i_TurId}")
required_modes = set(modes_list)

iteration_ids_with_required_modes = (
    otp_candidates_df.groupby("iteration_id")["mode"]
    .apply(lambda modes: required_modes.issubset(set(modes)))
)
iteration_ids_with_required_modes

iteration_id
-6    True
-5    True
-4    True
-3    True
-2    True
-1    True
 0    True
 1    True
 2    True
 3    True
Name: mode, dtype: bool

In [64]:
otp_candidates_df = otp_candidates_df[
    otp_candidates_df["iteration_id"].isin(
        iteration_ids_with_required_modes[
            iteration_ids_with_required_modes
        ].index
    )
].reset_index(drop=True)

In [65]:
if otp_candidates_df.empty:
    print(f"No OTP trips include all required transit modes ({modes_list}) for TurId: {i_TurId}")
otp_candidates_df

,iteration_id,leg_id,start_trip,end_trip,start_leg,end_leg,mode,route_short_name,distance_km,duration_min,waiting_time_min,from,to,generalized_cost,leg_geometry,system_notice_tag,system_notice_text,start_dt,waitingtime
0,-6,0,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524221000,1717524900000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:03:41+02:00,NaN
1,-6,1,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717524900000,1717525140000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:03:41+02:00,0.00
2,-6,2,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525140000,1717525455000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:03:41+02:00,0.00
3,-6,3,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717525740000,1717527000000,RAIL,IC,31.208,21,0.0,København H,Roskilde St.,2145,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:03:41+02:00,4.75
4,-6,4,2024-06-04T20:03:41+02:00,2024-06-04T21:13:32+02:00,1717527000000,1717528412000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:03:41+02:00,0.00
5,-5,0,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717525421000,1717526100000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 20:23:41+02:00,NaN
6,-5,1,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526100000,1717526340000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 20:23:41+02:00,0.00
7,-5,2,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526340000,1717526655000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 20:23:41+02:00,0.00
8,-5,3,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717526820000,1717528020000,RAIL,RE,31.426,20,0.0,København H,Roskilde St.,1965,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 20:23:41+02:00,2.75
9,-5,4,2024-06-04T20:23:41+02:00,2024-06-04T21:30:32+02:00,1717528020000,1717529432000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 20:23:41+02:00,0.00


In [68]:
import find_similar_trip
importlib.reload(find_similar_trip)
from find_similar_trip import find_similar_trip

time_based_match = find_similar_trip(
    tu_tur_row,
    otp_candidates_df,
    arrival_dev_weight=1,
    print_devation_details=True
)
time_based_match

Best matching trip: iteration_id = -2
Deviation details:
   iteration_id  depart_deviation  arrival_deviation  total_deviation
4            -2        -11.316667          -9.466667        20.783333
5            -1          1.683333          21.533333        23.216667
6             0         13.683333          21.533333        35.216667
7             1         21.683333          22.533333        44.216667
8             2         23.933333          22.533333        46.466667
3            -3        -36.316667         -29.466667        65.783333
9             3         31.683333          52.533333        84.216667
2            -4        -46.316667         -38.466667        84.783333
1            -5        -51.316667         -52.466667       103.783333
0            -6        -71.316667         -69.466667       140.783333


,iteration_id,leg_id,start_trip,end_trip,start_leg,end_leg,mode,route_short_name,distance_km,duration_min,waiting_time_min,from,to,generalized_cost,leg_geometry,system_notice_tag,system_notice_text,start_dt,waitingtime,end_dt
20,-2,0,2024-06-04T21:03:41+02:00,2024-06-04T22:13:32+02:00,1717527821000,1717528500000,WALK,NaN,0.657,11,0.0,Origin,Kongens Nytorv St. (Metro),1249,"[(55.68415, 12.58761), (55.68367, 12.5872), (5...",[],[],2024-06-04 21:03:41+02:00,NaN,2024-06-04 22:13:32+02:00
21,-2,1,2024-06-04T21:03:41+02:00,2024-06-04T22:13:32+02:00,1717528500000,1717528740000,SUBWAY,M3,1.782,4,0.0,Kongens Nytorv St. (Metro),København H (Metro),840,"[(55.67961, 12.58469), (55.67927, 12.5838), (5...",[],[],2024-06-04 21:03:41+02:00,0.00,2024-06-04 22:13:32+02:00
22,-2,2,2024-06-04T21:03:41+02:00,2024-06-04T22:13:32+02:00,1717528740000,1717529055000,WALK,NaN,0.218,5,0.0,København H (Metro),København H,497,"[(55.67194, 12.56411), (55.67194, 12.56413), (...",[],[],2024-06-04 21:03:41+02:00,0.00,2024-06-04 22:13:32+02:00
23,-2,3,2024-06-04T21:03:41+02:00,2024-06-04T22:13:32+02:00,1717529340000,1717530600000,RAIL,IC,31.208,21,0.0,København H,Roskilde St.,2145,"[(55.67275, 12.56489), (55.67219, 12.56544), (...",[],[],2024-06-04 21:03:41+02:00,4.75,2024-06-04 22:13:32+02:00
24,-2,4,2024-06-04T21:03:41+02:00,2024-06-04T22:13:32+02:00,1717530600000,1717532012000,WALK,NaN,1.795,24,0.0,Roskilde St.,Destination,3061,"[(55.63908, 12.08854), (55.63905, 12.08858), (...",[],[],2024-06-04 21:03:41+02:00,0.00,2024-06-04 22:13:32+02:00


In [69]:
if time_based_match is None:
    print(f"No best trip found for TurId: {i_TurId}")
time_based_match["TurId"] = i_TurId
time_based_match_print_col = ["mode", "distance_km", "waitingtime", "duration_min", "route_short_name", "from", "to"]
print("time_based_match:")
print(time_based_match[time_based_match_print_col].to_string(index=False, max_colwidth=None))




time_based_match:
  mode  distance_km  waitingtime  duration_min route_short_name                       from                         to
  WALK        0.657          NaN            11              NaN                     Origin Kongens Nytorv St. (Metro)
SUBWAY        1.782         0.00             4               M3 Kongens Nytorv St. (Metro)        København H (Metro)
  WALK        0.218         0.00             5              NaN        København H (Metro)                København H
  RAIL       31.208         4.75            21               IC                København H               Roskilde St.
  WALK        1.795         0.00            24              NaN               Roskilde St.                Destination
